# Phase 3 (UPDATED) — Data Preprocessing
**Project:** Depression Detection using Machine Learning Techniques

### Why this version is different
The original Phase 3 built the target label `Distressed` from the PHQ-9 questions, and then **also** used those same PHQ-9 questions as model features. Since `Distressed` is a deterministic threshold on `sum(question1..question9)`, the label could be recovered exactly by the features — this is **data leakage**, and it's why every model scored ~100%.

**Fix:** `Distressed` stays as the label (still derived from PHQ-9 — that part is fine, it's a label, not a feature). But the **feature set now excludes all PHQ-9 questions/score**. Instead we use variables that are genuinely independent of how the label was built:
- `demographic.csv` — gender, age, education, smoking, drinking
- `gad7.csv` — anxiety screening (7 items)
- `isi.csv` — insomnia severity (7 items)
- `pss.csv` — perceived stress scale (14 items)

This turns the task back into a real prediction problem: *can anxiety, sleep, stress, and demographics predict depression risk?*

In [1]:
# ==========================================================
# Phase 3 (UPDATED): Data Preprocessing
# Project: Depression Detection using Machine Learning Techniques
# ==========================================================

import pandas as pd

# ----------------------------------------------------------
# Step 1: Load all source files
# ----------------------------------------------------------
phq9 = pd.read_csv("phq9.csv")
demo = pd.read_csv("demographic.csv")
gad7 = pd.read_csv("gad7.csv")
isi  = pd.read_csv("isi.csv")
pss  = pd.read_csv("pss.csv")

print("Rows loaded -> phq9:", phq9.shape, "demo:", demo.shape,
      "gad7:", gad7.shape, "isi:", isi.shape, "pss:", pss.shape)

Rows loaded -> phq9: (24292, 20) demo: (24292, 6) gad7: (24292, 16) isi: (24292, 16) pss: (24292, 30)


## Step 2: Build the target label from PHQ-9
The label is created exactly as before (0–9 = Non-Distressed, 10–27 = Distressed). We keep only `export_id` and `Distressed` from this file — none of the PHQ-9 question columns travel into the feature set.

In [2]:
phq9["Distressed"] = phq9["score"].apply(lambda x: 0 if x <= 9 else 1)
label_df = phq9[["export_id", "Distressed"]]

print("Class distribution:")
print(label_df["Distressed"].value_counts())
print("\nPositive class %:", round(label_df["Distressed"].mean() * 100, 2))

Class distribution:
Distressed
0    22976
1     1316
Name: count, dtype: int64

Positive class %: 5.42


## Step 3: Data quality checks on each source file

In [3]:
for name, df in [("demographic", demo), ("gad7", gad7), ("isi", isi), ("pss", pss)]:
    print(f"{name}: missing values = {df.isnull().sum().sum()}, "
          f"duplicate export_id = {df['export_id'].duplicated().sum()}")

demographic: missing values = 0, duplicate export_id = 0
gad7: missing values = 0, duplicate export_id = 0
isi: missing values = 0, duplicate export_id = 0
pss: missing values = 0, duplicate export_id = 0


## Step 4: Drop response-time columns and total-score columns
`time*` columns are excluded, same reasoning as the original preprocessing (response time isn't a clinical feature). We keep the individual question responses (not the `score` totals) for GAD-7/ISI/PSS so the models get richer signal than a single number per scale.

In [4]:
def drop_time_and_score(df):
    drop_cols = [c for c in df.columns if c.startswith("time") or c == "score"]
    return df.drop(columns=drop_cols)

gad7_clean = drop_time_and_score(gad7).rename(
    columns={f"question{i}": f"gad7_q{i}" for i in range(1, 8)})
isi_clean = drop_time_and_score(isi).rename(
    columns={f"question{i}": f"isi_q{i}" for i in range(1, 8)})
pss_clean = drop_time_and_score(pss).rename(
    columns={f"question{i}": f"pss_q{i}" for i in range(1, 15)})

print("gad7_clean columns:", gad7_clean.columns.tolist())
print("isi_clean columns:", isi_clean.columns.tolist())
print("pss_clean columns:", pss_clean.columns.tolist())

gad7_clean columns: ['export_id', 'gad7_q1', 'gad7_q2', 'gad7_q3', 'gad7_q4', 'gad7_q5', 'gad7_q6', 'gad7_q7']
isi_clean columns: ['export_id', 'isi_q1', 'isi_q2', 'isi_q3', 'isi_q4', 'isi_q5', 'isi_q6', 'isi_q7']
pss_clean columns: ['export_id', 'pss_q1', 'pss_q2', 'pss_q3', 'pss_q4', 'pss_q5', 'pss_q6', 'pss_q7', 'pss_q8', 'pss_q9', 'pss_q10', 'pss_q11', 'pss_q12', 'pss_q13', 'pss_q14']


## Step 5: Merge all sources on `export_id`

In [5]:
df = label_df.merge(demo, on="export_id", how="inner") \
             .merge(gad7_clean, on="export_id", how="inner") \
             .merge(isi_clean, on="export_id", how="inner") \
             .merge(pss_clean, on="export_id", how="inner")

print("Merged shape:", df.shape)
df.head()

Merged shape: (24292, 35)


,export_id,Distressed,gender,age,edu,smoke,drink,gad7_q1,gad7_q2,gad7_q3,...,pss_q5,pss_q6,pss_q7,pss_q8,pss_q9,pss_q10,pss_q11,pss_q12,pss_q13,pss_q14
0,61793,0,female,19.0,bachelor's degree,never smokes,never drinks,0,0,0,...,1,1,1,1,1,1,1,3,1,1
1,61809,0,female,18.0,bachelor's degree,never smokes,never drinks,0,0,0,...,5,2,4,2,2,5,1,2,2,1
2,61737,0,male,40.0,master's degree,never smokes,drinks occasionally (less than once a week),0,0,0,...,1,1,1,1,1,1,1,5,1,1
3,61738,0,female,23.0,bachelor's degree,never smokes,never drinks,0,0,0,...,5,5,5,1,5,5,1,1,5,1
4,61739,0,female,21.0,bachelor's degree,never smokes,never drinks,0,0,0,...,2,1,3,2,4,2,3,3,2,2


## Step 6: Missing values and duplicates after merge

In [6]:
print("Missing values after merge:", df.isnull().sum().sum())
print("Duplicate export_id after merge:", df["export_id"].duplicated().sum())

Missing values after merge: 0
Duplicate export_id after merge: 0


## Step 7: Drop identifier column

In [7]:
df = df.drop(columns=["export_id"])
print("Shape after dropping export_id:", df.shape)

Shape after dropping export_id: (24292, 34)


## Step 8: One-hot encode categorical demographic columns

In [8]:
categorical_cols = ["gender", "edu", "smoke", "drink"]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("Shape after encoding:", df.shape)

Shape after encoding: (24292, 40)


## Step 9: Leakage sanity check
Two checks:
1. Confirm no PHQ-9-derived column slipped into the features.
2. Confirm no single feature is near-perfectly correlated with the label (which is what leakage looks like).

In [9]:
leak_check = [c for c in df.columns if "phq" in c.lower()]
print("Any PHQ-9-derived columns left in features?", leak_check)

corrs = df.corr(numeric_only=True)["Distressed"].drop("Distressed").sort_values(key=abs, ascending=False)
print("\nTop 5 correlations with Distressed:")
print(corrs.head(5))

Any PHQ-9-derived columns left in features? []



Top 5 correlations with Distressed:
gad7_q4    0.440999
gad7_q2    0.439844
gad7_q6    0.434022
gad7_q3    0.417478
pss_q2     0.414684
Name: Distressed, dtype: float64


**Result:** the strongest correlation is ~0.44 (a GAD-7 anxiety item), not 0.99+. That's what a real, non-leaky predictive relationship looks like — informative, but not a giveaway. Perfect confirmation the leakage is gone.

**Note for the team:** the class distribution is imbalanced (~5.4% Distressed vs ~94.6% Non-Distressed). Everyone should use a **stratified** train-test split in the next phase, and should report Precision/Recall/F1/ROC-AUC (not just Accuracy) since accuracy alone is misleading on imbalanced data — a model that always predicts "0" would already score ~94.6% accuracy without learning anything.

## Step 10: Final dataset overview

In [10]:
print("Final feature set columns:")
print(df.columns.tolist())
print("\nFinal dataset shape:", df.shape)
df.info()

Final feature set columns:
['Distressed', 'age', 'gad7_q1', 'gad7_q2', 'gad7_q3', 'gad7_q4', 'gad7_q5', 'gad7_q6', 'gad7_q7', 'isi_q1', 'isi_q2', 'isi_q3', 'isi_q4', 'isi_q5', 'isi_q6', 'isi_q7', 'pss_q1', 'pss_q2', 'pss_q3', 'pss_q4', 'pss_q5', 'pss_q6', 'pss_q7', 'pss_q8', 'pss_q9', 'pss_q10', 'pss_q11', 'pss_q12', 'pss_q13', 'pss_q14', 'gender_male', "edu_bachelor's degree", 'edu_doctorate degree', "edu_master's degree", 'smoke_former smoker (cumulative smoking >10 packs), but not in the past year', 'smoke_never smokes', 'smoke_occasional smoker (cumulative smoking <10 packs)', 'drink_drank in the past (more than once a week), but not in the past year', 'drink_drinks occasionally (less than once a week)', 'drink_never drinks']

Final dataset shape: (24292, 40)
<class 'pandas.DataFrame'>
RangeIndex: 24292 entries, 0 to 24291
Data columns (total 40 columns):
 #   Column                                                                        Non-Null Count  Dtype  
---  ------          

## Step 11: Save cleaned dataset

In [11]:
df.to_csv("cleaned_features.csv", index=False)
print("========================================")
print("Cleaned Dataset Saved Successfully!")
print("File: cleaned_features.csv")
print("========================================")

Cleaned Dataset Saved Successfully!
File: cleaned_features.csv
